In [2]:
# =====================================================================
# FINAL TRAINING: MELATIH MODEL DENGAN 100% DATA UNTUK DEPLOYMENT
# =====================================================================
import os
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from lightgbm import LGBMRegressor

# 1. Muat 100% Dataset (Tanpa Train-Test Split)
file_path = r"D:\LENTERA_LAUT\data\processed\final_dataset.csv"
df = pd.read_csv(file_path)

features = [
    "wave_height", "wind_speed_10m", "ocean_current_velocity", 
    "sea_surface_temperature", "precipitation", "visibility"
]
base_df = df.drop(columns=["time", "location", "id_location"], errors='ignore')

# 2. Direktori Khusus Model Produksi
production_dir = r"D:\LENTERA_LAUT\models\production_models"
os.makedirs(production_dir, exist_ok=True)

# 3. Kamus Parameter Pemenang Mutlak dari Hasil Optuna Terbaru
final_architectures = {
    "wave_height": LinearRegression(
        fit_intercept=True
    ),
    "wind_speed_10m": ExtraTreesRegressor(
        n_estimators=175, max_depth=12, min_samples_split=16, min_samples_leaf=7, 
        random_state=42, n_jobs=-1
    ),
    "ocean_current_velocity": ExtraTreesRegressor(
        n_estimators=174, max_depth=9, min_samples_split=11, min_samples_leaf=2, 
        random_state=42, n_jobs=-1
    ),
    "sea_surface_temperature": ExtraTreesRegressor(
        n_estimators=191, max_depth=11, min_samples_split=8, min_samples_leaf=6, 
        random_state=42, n_jobs=-1
    ),
    "precipitation": RandomForestRegressor(
        n_estimators=290, max_depth=12, min_samples_split=20, min_samples_leaf=10, 
        random_state=42, n_jobs=-1
    ),
    "visibility": LGBMRegressor(
        n_estimators=191, learning_rate=0.06097839109531514, max_depth=3, num_leaves=38, 
        reg_alpha=1.2255876808378807, reg_lambda=0.18825578416799568, 
        random_state=42, n_jobs=-1, verbose=-1
    )
}

print("🚀 MEMULAI PELATIHAN MODEL PRODUKSI (100% DATA)...")

# 4. Looping Pelatihan Final
for target in features:
    print(f"   -> Melatih model final untuk: {target.upper()}")
    
    # Pisahkan target dari fitur (X dan y menggunakan 100% baris)
    other_targets = [col for col in features if col != target]
    X_full = base_df.drop(columns=other_targets + [target], errors='ignore')
    y_full = base_df[target]
    
    # Ambil arsitektur model
    final_model = final_architectures[target]
    
    # Latih dengan seluruh data
    final_model.fit(X_full, y_full)
    
    # Ekspor sebagai Production Model
    model_filename = os.path.join(production_dir, f"production_{target}.pkl")
    joblib.dump(final_model, model_filename)
    
print(f"\n✅ SELESAI! Keenam model produksi siap digunakan oleh aplikasi dari direktori: {production_dir}")

🚀 MEMULAI PELATIHAN MODEL PRODUKSI (100% DATA)...
   -> Melatih model final untuk: WAVE_HEIGHT
   -> Melatih model final untuk: WIND_SPEED_10M
   -> Melatih model final untuk: OCEAN_CURRENT_VELOCITY
   -> Melatih model final untuk: SEA_SURFACE_TEMPERATURE
   -> Melatih model final untuk: PRECIPITATION
   -> Melatih model final untuk: VISIBILITY

✅ SELESAI! Keenam model produksi siap digunakan oleh aplikasi dari direktori: D:\LENTERA_LAUT\models\production_models
